
# **5장 단순 최저 운송사 선정_기준정보변경**
  
※ 기준정보변경
- 문제1: 단순한 제약조건을 고려하는 최저 운송사 선정 문제
- 2개의 함수로 구성된 전체 코드와 세부 기능을 확인하는 상세 코드로 구성되어 있음
- 코드생성 프롬프트에 코드에 대한 주석처리 요청 추가
- 수리모델을 풀어쓰기 함수 생성 추가
- 50개 운송사 기준정보 변경 및 상세 코드설명 시 상세출력함수 비 활성화

# 교재의 PuLP 3.3.0 버전 고정 설치하여 실습한다.
  - 2026년 최신 PuLP 4.0.0으로 변경되면서 교재의 기존 코드와 API 호환성 문제가 발생한다.

In [ ]:
# ==================================================
# PuLP 3.3.0 버전 고정 설치
# ==================================================

%pip install --force-reinstall "pulp==3.3.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 54.6 MB/s eta 0:00:00
  Attempting uninstall: pulp
    Found existing installation: PuLP 4.0.0
    Uninstalling PuLP-4.0.0:
      Successfully uninstalled PuLP-4.0.0


In [ ]:
# pip install pulp

In [ ]:
# 필요한 라이브러리 불러오기
import pandas as pd  # 엑셀 데이터를 다루기 위한 라이브러리
import time  # 수행 시간 측정용
import pulp  # 선형계획법 최적화 라이브러리
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, LpInteger, value  # PuLP에서 자주 쓰는 클래스 및 함수들

# 최적화 함수 정의
def optimize_simple_assignment():
    start_time = time.time()  # 시작 시간 기록

    #--------------------------------------------------------
    # 1) 엑셀에서 데이터 읽어오기
    #--------------------------------------------------------
    input_file = "/content/sample_data/50개운송사_기준정보.xlsx"  # 입력 엑셀 파일 경로

    # 운송사 관련 데이터 읽기 (A~E열: 운송사, 운송구간, 트럭유형, 운송단가, 트럭대수)
    df = pd.read_excel(input_file, sheet_name=0, usecols="A:E")

    # 수요 데이터 읽기 (H~J열: 운송구간, 트럭유형, 필요대수)
    df_demand = pd.read_excel(input_file, sheet_name=0, usecols="H:J")
    df_demand.columns = ['운송구간', '트럭유형', '필요대수']  # 컬럼명 재정의

    # 데이터 타입 정리 및 결측치 제거
    df['운송단가'] = pd.to_numeric(df['운송단가'], errors='coerce')
    df['트럭대수'] = pd.to_numeric(df['트럭대수'], errors='coerce')
    df_demand['필요대수'] = pd.to_numeric(df_demand['필요대수'], errors='coerce')
    df.dropna(subset=['운송단가', '트럭대수'], inplace=True)
    df_demand.dropna(subset=['필요대수'], inplace=True)

    # 무한대 값(Inf, -Inf)이 있는 데이터도 제거
    df = df[(~df['운송단가'].isin([float('inf'), float('-inf')])) &
        (~df['트럭대수'].isin([float('inf'), float('-inf')]))]
    df_demand = df_demand[(~df_demand['필요대수'].isin([float('inf'), float('-inf')]))]

    #--------------------------------------------------------
    # 2) 파라미터 구성 (딕셔너리 형태로 정리)
    #--------------------------------------------------------
    cost_dict = {}      # 운송단가: {(운송사, 구간, 트럭유형): 단가}
    capacity_dict = {}  # 운송사별 공급 가능한 트럭 수: {(운송사, 구간, 트럭유형): 대수}
    # 데이터프레임 df에서 각 행(row)을 읽어와 딕셔너리에 저장합니다.
    for _, row in df.iterrows():
        key = (row['운송사'], row['운송구간'], row['트럭유형'])  # 딕셔너리의 키로 사용할 튜플
        cost_dict[key] = row['운송단가']  # 운송단가 저장
        capacity_dict[key] = row['트럭대수']  # 트럭대수 저장

    demand_dict = {}  # 수요 트럭 수: {(구간, 트럭유형): 필요대수}
    for _, row in df_demand.iterrows():
        demand_dict[(row['운송구간'], row['트럭유형'])] = row['필요대수']

    truckers = df['운송사'].unique()  # 운송사 목록 추출
    route_type_list = demand_dict.keys()  # 수요 조건에 해당하는 (구간, 트럭유형) 조합

    #--------------------------------------------------------
    # 3) PuLP 최적화 모델 생성
    #--------------------------------------------------------
    prob = LpProblem("Simple_Optimization", LpMinimize)  # 목적: 비용 최소화

    # 결정변수 정의: x_(운송사)_(구간)_(트럭유형) = 몇 대 배정할지
    x_vars = {}
    for (i, r, t), cost in cost_dict.items():
        x_vars[(i, r, t)] = LpVariable(f"x_{i}_{r}_{t}", lowBound=0, cat=LpInteger)  # 정수변수로 정의

    #--------------------------------------------------------
    # 4) 목적함수 정의: 전체 운송비용 최소화
    #--------------------------------------------------------
    prob += lpSum([
        cost_dict[(i, r, t)] * x_vars[(i, r, t)]
        for (i, r, t) in cost_dict
    ]), "Minimize_TotalCost"

    #--------------------------------------------------------
    # 5) 제약조건 정의
    #--------------------------------------------------------

    # 제약조건 1: 각 (구간, 트럭유형)에 대해 수요만큼 트럭이 배정되어야 함
    for (r, t) in route_type_list:
        prob += lpSum([
            x_vars[(i, r, t)]
            for i in truckers if (i, r, t) in x_vars
        ]) == demand_dict[(r, t)], f"Demand_{r}_{t}"

    # 제약조건 2: 운송사별 보유 트럭 수를 초과하지 않도록
    for (i, r, t), cap in capacity_dict.items():
        prob += x_vars[(i, r, t)] <= cap, f"Capacity_{i}_{r}_{t}"

    #--------------------------------------------------------
    # 6) 최적화 실행
    #--------------------------------------------------------
    prob.solve(pulp.PULP_CBC_CMD(msg=0))  # CBC Solver를 사용하여 최적화 수행

    #--------------------------------------------------------
    # 7) 결과 출력
    #--------------------------------------------------------
    print("Status:", pulp.LpStatus[prob.status])  # 최적화 상태 출력
    if prob.status not in [1, 2]:  # Optimal(1) 또는 Feasible(2)인 경우만 유효
        print("=> 유효한 해를 찾지 못했습니다.")
        return None

    # 총 운송비 계산
    total_cost = value(prob.objective)
    print("총 운송비:", total_cost)

    # 최적화 결과(트럭 배정 현황) 정리
    result_data = []
    for (i, r, t), var in x_vars.items():
        if var.varValue > 0:  # 배정된 트럭 수가 0보다 큰 경우만 출력
            result_data.append([i, r, t, cost_dict[(i, r, t)], int(var.varValue)])

    # 결과를 데이터프레임으로 출력
    df_result = pd.DataFrame(result_data, columns=['운송사', '운송구간', '트럭유형', '운송단가', '선택된트럭수'])
    print("\n--- 배정 결과 ---")
    print(df_result.to_string(index=False))  # 결과 출력

    # 엑셀로 저장
    output_file = "/content/sample_data/단순최적화결과_50개운송사.xlsx"
    df_result.to_excel(output_file, index=False)

    end_time = time.time()  # 종료 시간 기록
    print("\n프로그램 수행시간(초):", round(end_time - start_time, 4))  # 수행 시간 출력

    return prob  # 최적화 모델 객체 반환

# 최적화 모델 상세 출력 함수
def display_optimization_model(prob):
    """
    최적화 모델의 목적식과 제약식을 출력하는 함수
    """
    print("\n=== 목적함수 ===")
    print(prob.objective)  # 목적함수 출력

    print("\n=== 제약조건 ===")
    for name, constraint in prob.constraints.items():  # 제약조건 모두 출력
        print(f"{name}: {constraint}")

    print("\n=== 결정변수 ===")
    for var in prob.variables():  # 변수명과 해 출력
        print(f"{var.name}: {var.varValue}")

# 메인 실행부
if __name__ == "__main__":
    # 최적화 실행
    prob = optimize_simple_assignment()

    # 모델 내용 출력
    # if prob is not None:
        # display_optimization_model(prob)


Status: Optimal
총 운송비: 697410.0

--- 배정 결과 ---
  운송사 운송구간 트럭유형  운송단가  선택된트럭수
운송사01 구간37 트럭03   117      30
운송사02 구간11 트럭02   152      30
운송사02 구간15 트럭01   142      30
운송사02 구간38 트럭01   166      30
운송사03 구간04 트럭02   102      30
운송사03 구간12 트럭02   102      30
운송사03 구간31 트럭04   117      30
운송사03 구간46 트럭04   104      30
운송사03 구간50 트럭04   105      30
운송사04 구간13 트럭03   103      30
운송사04 구간14 트럭01   162      30
운송사04 구간21 트럭04   120      30
운송사04 구간24 트럭01   103      30
운송사04 구간24 트럭02   120      30
운송사04 구간38 트럭04   100      30
운송사04 구간40 트럭02   121      30
운송사05 구간10 트럭02   130      30
운송사06 구간01 트럭04   108      30
운송사06 구간05 트럭03   103      30
운송사06 구간25 트럭01   154      30
운송사06 구간43 트럭04   115      30
운송사06 구간48 트럭01   101      30
운송사07 구간02 트럭03   155      30
운송사07 구간05 트럭01   117      30
운송사07 구간12 트럭03   110      30
운송사07 구간32 트럭02   115      30
운송사07 구간34 트럭04   140      30
운송사07 구간47 트럭04   100      30
운송사08 구간10 트럭03   126      30
운송사08 구간32 트럭04   105      30
운송사09 구간07 트럭04   120  

# 최적함 함수 정의
def optimize_simple_assignment():

In [ ]:
start_time = time.time()  # 시작 시간 기록

# 기준정보의 운송단가와 공급제약 데이터 읽기

In [ ]:
    input_file = "/content/sample_data/50개운송사_기준정보.xlsx"  # 입력 엑셀 파일 경로
    # 운송사 관련 데이터 읽기 (A~E열: 운송사, 운송구간, 트럭유형, 운송단가, 트럭대수)
    df = pd.read_excel(input_file, sheet_name=0, usecols="A:E")

In [ ]:
df

,운송사,운송구간,트럭유형,운송단가,트럭대수
0,운송사01,구간01,트럭01,785,42
1,운송사01,구간01,트럭02,886,69
2,운송사01,구간01,트럭03,905,38
3,운송사01,구간01,트럭04,921,48
4,운송사01,구간02,트럭01,294,53
...,...,...,...,...,...
9995,운송사50,구간49,트럭04,589,38
9996,운송사50,구간50,트럭01,417,54
9997,운송사50,구간50,트럭02,199,55
9998,운송사50,구간50,트럭03,319,30


#도표 5-24. 기준정보의 수요 데이터 읽기

In [ ]:
    # 수요 데이터 읽기 (H~J열: 운송구간, 트럭유형, 필요대수)
    df_demand = pd.read_excel(input_file, sheet_name=0, usecols="H:J")
    df_demand.columns = ['운송구간', '트럭유형', '필요대수']  # 컬럼명 재정의

In [ ]:
df_demand

,운송구간,트럭유형,필요대수
0,구간01,트럭01,30.0
1,구간01,트럭02,30.0
2,구간01,트럭03,30.0
3,구간01,트럭04,30.0
4,구간02,트럭01,30.0
...,...,...,...
9995,NaN,NaN,NaN
9996,NaN,NaN,NaN
9997,NaN,NaN,NaN
9998,NaN,NaN,NaN


# 도표 5-25. 결측치 제거

In [ ]:
    # 데이터 타입 정리 및 결측치 제거
    df['운송단가'] = pd.to_numeric(df['운송단가'], errors='coerce')
    df['트럭대수'] = pd.to_numeric(df['트럭대수'], errors='coerce')
    df_demand['필요대수'] = pd.to_numeric(df_demand['필요대수'], errors='coerce')
    df.dropna(subset=['운송단가', '트럭대수'], inplace=True)
    df_demand.dropna(subset=['필요대수'], inplace=True)

In [ ]:
df_demand

,운송구간,트럭유형,필요대수
0,구간01,트럭01,30.0
1,구간01,트럭02,30.0
2,구간01,트럭03,30.0
3,구간01,트럭04,30.0
4,구간02,트럭01,30.0
...,...,...,...
195,구간49,트럭04,30.0
196,구간50,트럭01,30.0
197,구간50,트럭02,30.0
198,구간50,트럭03,30.0


# 도표 5-28. 무한대 값 제거 파이썬 코드와 예시

In [ ]:
# 무한대 값(Inf, -Inf)이 있는 데이터도 제거합니다.
df = df[(~df['운송단가'].isin([float('inf'), float('-inf')])) &
        (~df['트럭대수'].isin([float('inf'), float('-inf')]))]

df_demand = df_demand[(~df_demand['필요대수'].isin([float('inf'), float('-inf')]))]

In [ ]:
df

,운송사,운송구간,트럭유형,운송단가,트럭대수
0,운송사01,구간01,트럭01,785,42
1,운송사01,구간01,트럭02,886,69
2,운송사01,구간01,트럭03,905,38
3,운송사01,구간01,트럭04,921,48
4,운송사01,구간02,트럭01,294,53
...,...,...,...,...,...
9995,운송사50,구간49,트럭04,589,38
9996,운송사50,구간50,트럭01,417,54
9997,운송사50,구간50,트럭02,199,55
9998,운송사50,구간50,트럭03,319,30


# 딕셔너리 저장

In [ ]:
    cost_dict = {}      # 운송단가: {(운송사, 구간, 트럭유형): 단가}
    capacity_dict = {}  # 운송사별 공급 가능한 트럭 수: {(운송사, 구간, 트럭유형): 대수}
    # 데이터프레임 df에서 각 행(row)을 읽어와 딕셔너리에 저장합니다.
    for _, row in df.iterrows():
        key = (row['운송사'], row['운송구간'], row['트럭유형'])  # 딕셔너리의 키로 사용할 튜플
        cost_dict[key] = row['운송단가']  # 운송단가 저장
        capacity_dict[key] = row['트럭대수']  # 트럭대수 저장

    print(cost_dict)
    print(capacity_dict)

{('운송사01', '구간01', '트럭01'): 785, ('운송사01', '구간01', '트럭02'): 886, ('운송사01', '구간01', '트럭03'): 905, ('운송사01', '구간01', '트럭04'): 921, ('운송사01', '구간02', '트럭01'): 294, ('운송사01', '구간02', '트럭02'): 829, ('운송사01', '구간02', '트럭03'): 666, ('운송사01', '구간02', '트럭04'): 309, ('운송사01', '구간03', '트럭01'): 695, ('운송사01', '구간03', '트럭02'): 277, ('운송사01', '구간03', '트럭03'): 841, ('운송사01', '구간03', '트럭04'): 753, ('운송사01', '구간04', '트럭01'): 310, ('운송사01', '구간04', '트럭02'): 908, ('운송사01', '구간04', '트럭03'): 950, ('운송사01', '구간04', '트럭04'): 287, ('운송사01', '구간05', '트럭01'): 483, ('운송사01', '구간05', '트럭02'): 501, ('운송사01', '구간05', '트럭03'): 834, ('운송사01', '구간05', '트럭04'): 681, ('운송사01', '구간06', '트럭01'): 650, ('운송사01', '구간06', '트럭02'): 249, ('운송사01', '구간06', '트럭03'): 720, ('운송사01', '구간06', '트럭04'): 358, ('운송사01', '구간07', '트럭01'): 485, ('운송사01', '구간07', '트럭02'): 502, ('운송사01', '구간07', '트럭03'): 168, ('운송사01', '구간07', '트럭04'): 931, ('운송사01', '구간08', '트럭01'): 606, ('운송사01', '구간08', '트럭02'): 663, ('운송사01', '구간08', '트럭03'): 940, ('운송사01

In [ ]:
    demand_dict = {}  # 수요 트럭 수: {(구간, 트럭유형): 필요대수}
    for _, row in df_demand.iterrows():
        demand_dict[(row['운송구간'], row['트럭유형'])] = row['필요대수']

    truckers = df['운송사'].unique()  # 운송사 목록 추출
    route_type_list = demand_dict.keys()  # 수요 조건에 해당하는 (구간, 트럭유형) 조합

In [ ]:
truckers, route_type_list

(array(['운송사01', '운송사02', '운송사03', '운송사04', '운송사05', '운송사06', '운송사07',
        '운송사08', '운송사09', '운송사10', '운송사11', '운송사12', '운송사13', '운송사14',
        '운송사15', '운송사16', '운송사17', '운송사18', '운송사19', '운송사20', '운송사21',
        '운송사22', '운송사23', '운송사24', '운송사25', '운송사26', '운송사27', '운송사28',
        '운송사29', '운송사30', '운송사31', '운송사32', '운송사33', '운송사34', '운송사35',
        '운송사36', '운송사37', '운송사38', '운송사39', '운송사40', '운송사41', '운송사42',
        '운송사43', '운송사44', '운송사45', '운송사46', '운송사47', '운송사48', '운송사49',
        '운송사50'], dtype=object),
 dict_keys([('구간01', '트럭01'), ('구간01', '트럭02'), ('구간01', '트럭03'), ('구간01', '트럭04'), ('구간02', '트럭01'), ('구간02', '트럭02'), ('구간02', '트럭03'), ('구간02', '트럭04'), ('구간03', '트럭01'), ('구간03', '트럭02'), ('구간03', '트럭03'), ('구간03', '트럭04'), ('구간04', '트럭01'), ('구간04', '트럭02'), ('구간04', '트럭03'), ('구간04', '트럭04'), ('구간05', '트럭01'), ('구간05', '트럭02'), ('구간05', '트럭03'), ('구간05', '트럭04'), ('구간06', '트럭01'), ('구간06', '트럭02'), ('구간06', '트럭03'), ('구간06', '트럭04'), ('구간07', '트럭01'), ('구간07',

# 도표 5-32. 최적화 모델의 객체 생성

In [ ]:
prob = LpProblem("Simple_Optimization", LpMinimize)  # 목적: 비용 최소화
print(prob)

Simple_Optimization:
MINIMIZE
None
VARIABLES



# 도표 5-33. 결정변수의 정의와 저장

In [ ]:
    # 결정변수 정의: x_(운송사)_(구간)_(트럭유형) = 몇 대 배정할지
    x_vars = {}
    for (i, r, t), cost in cost_dict.items():
        x_vars[(i, r, t)] = LpVariable(f"x_{i}_{r}_{t}", lowBound=0, cat=LpInteger)  # 정수변수로 정의
    print(x_vars)

{('운송사01', '구간01', '트럭01'): x_운송사01_구간01_트럭01, ('운송사01', '구간01', '트럭02'): x_운송사01_구간01_트럭02, ('운송사01', '구간01', '트럭03'): x_운송사01_구간01_트럭03, ('운송사01', '구간01', '트럭04'): x_운송사01_구간01_트럭04, ('운송사01', '구간02', '트럭01'): x_운송사01_구간02_트럭01, ('운송사01', '구간02', '트럭02'): x_운송사01_구간02_트럭02, ('운송사01', '구간02', '트럭03'): x_운송사01_구간02_트럭03, ('운송사01', '구간02', '트럭04'): x_운송사01_구간02_트럭04, ('운송사01', '구간03', '트럭01'): x_운송사01_구간03_트럭01, ('운송사01', '구간03', '트럭02'): x_운송사01_구간03_트럭02, ('운송사01', '구간03', '트럭03'): x_운송사01_구간03_트럭03, ('운송사01', '구간03', '트럭04'): x_운송사01_구간03_트럭04, ('운송사01', '구간04', '트럭01'): x_운송사01_구간04_트럭01, ('운송사01', '구간04', '트럭02'): x_운송사01_구간04_트럭02, ('운송사01', '구간04', '트럭03'): x_운송사01_구간04_트럭03, ('운송사01', '구간04', '트럭04'): x_운송사01_구간04_트럭04, ('운송사01', '구간05', '트럭01'): x_운송사01_구간05_트럭01, ('운송사01', '구간05', '트럭02'): x_운송사01_구간05_트럭02, ('운송사01', '구간05', '트럭03'): x_운송사01_구간05_트럭03, ('운송사01', '구간05', '트럭04'): x_운송사01_구간05_트럭04, ('운송사01', '구간06', '트럭01'): x_운송사01_구간06_트럭01, ('운송사01', '구간06', '트럭02'): x_운송사0

In [ ]:
    print(prob)

Simple_Optimization:
MINIMIZE
None
VARIABLES



# 도표 5-34. 목적식과 결정변수가 추가된 prob 객체

In [ ]:
    prob += lpSum([
        cost_dict[(i, r, t)] * x_vars[(i, r, t)]
        for (i, r, t) in cost_dict
    ]), "Minimize_TotalCost"
    print(prob)

Simple_Optimization:
MINIMIZE
785*x_운송사01_구간01_트럭01 + 886*x_운송사01_구간01_트럭02 + 905*x_운송사01_구간01_트럭03 + 921*x_운송사01_구간01_트럭04 + 294*x_운송사01_구간02_트럭01 + 829*x_운송사01_구간02_트럭02 + 666*x_운송사01_구간02_트럭03 + 309*x_운송사01_구간02_트럭04 + 695*x_운송사01_구간03_트럭01 + 277*x_운송사01_구간03_트럭02 + 841*x_운송사01_구간03_트럭03 + 753*x_운송사01_구간03_트럭04 + 310*x_운송사01_구간04_트럭01 + 908*x_운송사01_구간04_트럭02 + 950*x_운송사01_구간04_트럭03 + 287*x_운송사01_구간04_트럭04 + 483*x_운송사01_구간05_트럭01 + 501*x_운송사01_구간05_트럭02 + 834*x_운송사01_구간05_트럭03 + 681*x_운송사01_구간05_트럭04 + 650*x_운송사01_구간06_트럭01 + 249*x_운송사01_구간06_트럭02 + 720*x_운송사01_구간06_트럭03 + 358*x_운송사01_구간06_트럭04 + 485*x_운송사01_구간07_트럭01 + 502*x_운송사01_구간07_트럭02 + 168*x_운송사01_구간07_트럭03 + 931*x_운송사01_구간07_트럭04 + 606*x_운송사01_구간08_트럭01 + 663*x_운송사01_구간08_트럭02 + 940*x_운송사01_구간08_트럭03 + 623*x_운송사01_구간08_트럭04 + 530*x_운송사01_구간09_트럭01 + 372*x_운송사01_구간09_트럭02 + 537*x_운송사01_구간09_트럭03 + 601*x_운송사01_구간09_트럭04 + 210*x_운송사01_구간10_트럭01 + 547*x_운송사01_구간10_트럭02 + 420*x_운송사01_구간10_트럭03 + 243*x_운송사01_구간10_트럭04 + 225*x_운송사0

# 도표 5-36. 제약조건1 필요대수 충족

In [ ]:
    # 제약조건 1: 각 (구간, 트럭유형)에 대해 수요만큼 트럭이 배정되어야 함
    for (r, t) in route_type_list:
      prob += lpSum([
            x_vars[(i, r, t)]
            for i in truckers if (i, r, t) in x_vars
      ]) == demand_dict[(r, t)], f"Demand_{r}_{t}"

In [ ]:
print(prob)

Simple_Optimization:
MINIMIZE
785*x_운송사01_구간01_트럭01 + 886*x_운송사01_구간01_트럭02 + 905*x_운송사01_구간01_트럭03 + 921*x_운송사01_구간01_트럭04 + 294*x_운송사01_구간02_트럭01 + 829*x_운송사01_구간02_트럭02 + 666*x_운송사01_구간02_트럭03 + 309*x_운송사01_구간02_트럭04 + 695*x_운송사01_구간03_트럭01 + 277*x_운송사01_구간03_트럭02 + 841*x_운송사01_구간03_트럭03 + 753*x_운송사01_구간03_트럭04 + 310*x_운송사01_구간04_트럭01 + 908*x_운송사01_구간04_트럭02 + 950*x_운송사01_구간04_트럭03 + 287*x_운송사01_구간04_트럭04 + 483*x_운송사01_구간05_트럭01 + 501*x_운송사01_구간05_트럭02 + 834*x_운송사01_구간05_트럭03 + 681*x_운송사01_구간05_트럭04 + 650*x_운송사01_구간06_트럭01 + 249*x_운송사01_구간06_트럭02 + 720*x_운송사01_구간06_트럭03 + 358*x_운송사01_구간06_트럭04 + 485*x_운송사01_구간07_트럭01 + 502*x_운송사01_구간07_트럭02 + 168*x_운송사01_구간07_트럭03 + 931*x_운송사01_구간07_트럭04 + 606*x_운송사01_구간08_트럭01 + 663*x_운송사01_구간08_트럭02 + 940*x_운송사01_구간08_트럭03 + 623*x_운송사01_구간08_트럭04 + 530*x_운송사01_구간09_트럭01 + 372*x_운송사01_구간09_트럭02 + 537*x_운송사01_구간09_트럭03 + 601*x_운송사01_구간09_트럭04 + 210*x_운송사01_구간10_트럭01 + 547*x_운송사01_구간10_트럭02 + 420*x_운송사01_구간10_트럭03 + 243*x_운송사01_구간10_트럭04 + 225*x_운송사0

# 도표 5-38. 제약조건2 보유 트럭대수 초과 금지

In [ ]:
    # 제약조건 2: 운송사별 보유 트럭 수를 초과하지 않도록
    for (i, r, t), cap in capacity_dict.items():
        prob += x_vars[(i, r, t)] <= cap, f"Capacity_{i}_{r}_{t}"

In [ ]:
print(prob)

Simple_Optimization:
MINIMIZE
785*x_운송사01_구간01_트럭01 + 886*x_운송사01_구간01_트럭02 + 905*x_운송사01_구간01_트럭03 + 921*x_운송사01_구간01_트럭04 + 294*x_운송사01_구간02_트럭01 + 829*x_운송사01_구간02_트럭02 + 666*x_운송사01_구간02_트럭03 + 309*x_운송사01_구간02_트럭04 + 695*x_운송사01_구간03_트럭01 + 277*x_운송사01_구간03_트럭02 + 841*x_운송사01_구간03_트럭03 + 753*x_운송사01_구간03_트럭04 + 310*x_운송사01_구간04_트럭01 + 908*x_운송사01_구간04_트럭02 + 950*x_운송사01_구간04_트럭03 + 287*x_운송사01_구간04_트럭04 + 483*x_운송사01_구간05_트럭01 + 501*x_운송사01_구간05_트럭02 + 834*x_운송사01_구간05_트럭03 + 681*x_운송사01_구간05_트럭04 + 650*x_운송사01_구간06_트럭01 + 249*x_운송사01_구간06_트럭02 + 720*x_운송사01_구간06_트럭03 + 358*x_운송사01_구간06_트럭04 + 485*x_운송사01_구간07_트럭01 + 502*x_운송사01_구간07_트럭02 + 168*x_운송사01_구간07_트럭03 + 931*x_운송사01_구간07_트럭04 + 606*x_운송사01_구간08_트럭01 + 663*x_운송사01_구간08_트럭02 + 940*x_운송사01_구간08_트럭03 + 623*x_운송사01_구간08_트럭04 + 530*x_운송사01_구간09_트럭01 + 372*x_운송사01_구간09_트럭02 + 537*x_운송사01_구간09_트럭03 + 601*x_운송사01_구간09_트럭04 + 210*x_운송사01_구간10_트럭01 + 547*x_운송사01_구간10_트럭02 + 420*x_운송사01_구간10_트럭03 + 243*x_운송사01_구간10_트럭04 + 225*x_운송사0

# 최적화 실행

In [ ]:
 prob.solve(pulp.PULP_CBC_CMD(msg=0))  # CBC Solver를 사용하여 최적화 수행

1

# 결과 출력

In [ ]:
    print("Status:", pulp.LpStatus[prob.status])  # 최적화 상태 출력
    if prob.status not in [1, 2]:  # Optimal(1) 또는 Feasible(2)인 경우만 유효
        print("=> 유효한 해를 찾지 못했습니다.")
        # return None # optimize_simple_assignment( )내 코드가 아니므로 주석처리

    # 총 운송비 계산
    total_cost = value(prob.objective)
    print("총 운송비:", total_cost)

    # 최적화 결과(트럭 배정 현황) 정리
    result_data = []
    for (i, r, t), var in x_vars.items():
        if var.varValue > 0:  # 배정된 트럭 수가 0보다 큰 경우만 출력
            result_data.append([i, r, t, cost_dict[(i, r, t)], int(var.varValue)])

    # 결과를 데이터프레임으로 출력
    df_result = pd.DataFrame(result_data, columns=['운송사', '운송구간', '트럭유형', '운송단가', '선택된트럭수'])
    print("\n--- 배정 결과 ---")
    print(df_result.to_string(index=False))  # 결과 출력

    end_time = time.time()  # 종료 시간 기록
    print("\n프로그램 수행시간(초):", round(end_time - start_time, 4))  # 수행 시간 출력

    # return prob # 최적화 모델 객체 반환 # optimize_simple_assignment( )내 코드가 아니므로 주석처리


Status: Optimal
총 운송비: 697410.0

--- 배정 결과 ---
  운송사 운송구간 트럭유형  운송단가  선택된트럭수
운송사01 구간37 트럭03   117      30
운송사02 구간11 트럭02   152      30
운송사02 구간15 트럭01   142      30
운송사02 구간38 트럭01   166      30
운송사03 구간04 트럭02   102      30
운송사03 구간12 트럭02   102      30
운송사03 구간31 트럭04   117      30
운송사03 구간46 트럭04   104      30
운송사03 구간50 트럭04   105      30
운송사04 구간13 트럭03   103      30
운송사04 구간14 트럭01   162      30
운송사04 구간21 트럭04   120      30
운송사04 구간24 트럭01   103      30
운송사04 구간24 트럭02   120      30
운송사04 구간38 트럭04   100      30
운송사04 구간40 트럭02   121      30
운송사05 구간10 트럭02   130      30
운송사06 구간01 트럭04   108      30
운송사06 구간05 트럭03   103      30
운송사06 구간25 트럭01   154      30
운송사06 구간43 트럭04   115      30
운송사06 구간48 트럭01   101      30
운송사07 구간02 트럭03   155      30
운송사07 구간05 트럭01   117      30
운송사07 구간12 트럭03   110      30
운송사07 구간32 트럭02   115      30
운송사07 구간34 트럭04   140      30
운송사07 구간47 트럭04   100      30
운송사08 구간10 트럭03   126      30
운송사08 구간32 트럭04   105      30
운송사09 구간07 트럭04   120  

# 최적화 모델 상세 출력
def display_optimization_model(prob):

In [ ]:
    print("\n=== 목적함수 ===")
    print(prob.objective)  # 목적함수 출력

    print("\n=== 제약조건 ===")
    for name, constraint in prob.constraints.items():  # 제약조건 모두 출력
        print(f"{name}: {constraint}")

    print("\n=== 결정변수 ===")
    for var in prob.variables():  # 변수명과 해 출력
        print(f"{var.name}: {var.varValue}")

Streaming output truncated to the last 5000 lines.
x_운송사26_구간01_트럭01: 0.0
x_운송사26_구간01_트럭02: 0.0
x_운송사26_구간01_트럭03: 0.0
x_운송사26_구간01_트럭04: 0.0
x_운송사26_구간02_트럭01: 0.0
x_운송사26_구간02_트럭02: 0.0
x_운송사26_구간02_트럭03: 0.0
x_운송사26_구간02_트럭04: 0.0
x_운송사26_구간03_트럭01: 0.0
x_운송사26_구간03_트럭02: 0.0
x_운송사26_구간03_트럭03: 0.0
x_운송사26_구간03_트럭04: 0.0
x_운송사26_구간04_트럭01: 0.0
x_운송사26_구간04_트럭02: 0.0
x_운송사26_구간04_트럭03: 0.0
x_운송사26_구간04_트럭04: 0.0
x_운송사26_구간05_트럭01: 0.0
x_운송사26_구간05_트럭02: 0.0
x_운송사26_구간05_트럭03: 0.0
x_운송사26_구간05_트럭04: 0.0
x_운송사26_구간06_트럭01: 0.0
x_운송사26_구간06_트럭02: 0.0
x_운송사26_구간06_트럭03: 0.0
x_운송사26_구간06_트럭04: 0.0
x_운송사26_구간07_트럭01: 0.0
x_운송사26_구간07_트럭02: 0.0
x_운송사26_구간07_트럭03: 0.0
x_운송사26_구간07_트럭04: 0.0
x_운송사26_구간08_트럭01: 0.0
x_운송사26_구간08_트럭02: 0.0
x_운송사26_구간08_트럭03: 0.0
x_운송사26_구간08_트럭04: 0.0
x_운송사26_구간09_트럭01: 0.0
x_운송사26_구간09_트럭02: 0.0
x_운송사26_구간09_트럭03: 0.0
x_운송사26_구간09_트럭04: 0.0
x_운송사26_구간10_트럭01: 0.0
x_운송사26_구간10_트럭02: 0.0
x_운송사26_구간10_트럭03: 0.0
x_운송사26_구간10_트럭04: 0.0
x_운송사26_구간11_트럭01: 0.0
x_운송사2

In [ ]:
print(len(prob.constraints))  # 총 몇 개 제약조건이 정의되어 있는지 확인

10200
